# AI Shopping Assistant — LangGraph + Google Gemini

A Colab-ready mock shopping assistant using a fictional catalogue. It searches products, checks stock, calculates totals in Python, shows an order preview, and saves orders **only after explicit approval**.

**No real store, payment, or external checkout is used.** Orders are saved to `/content/mock_orders.json` for the current Colab runtime.

## Run order
Run cells from top to bottom. Enter your Gemini API key when prompted. If your Google AI account does not support the default model, change `GEMINI_MODEL` in Cell 3 to a model available to your key.

## 1. Install dependencies

In [1]:
%pip -q install -U langgraph langchain-google-genai langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.6/259.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 2. Configure Gemini securely
Use `getpass` so the key is not echoed in notebook output. Do not hard-code or share your API key.

In [2]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("Gemini API key configured for this runtime.")

Enter your Google Gemini API key: ··········
Gemini API key configured for this runtime.


## 3. Imports and fictional catalogue

In [8]:
import json
import re
import uuid
from pathlib import Path
from datetime import datetime, timezone
from typing import TypedDict, Optional

from IPython.display import display, Markdown
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# Fallback to gemini-2.5-flash as gemini-3.8-flash is experiencing high demand.
GEMINI_MODEL = "gemini-2.5-flash"

PRODUCTS = {
    "P001": {"name": "Basic Wireless Mouse", "price": 500, "stock": 10, "category": "wireless mouse"},
    "P002": {"name": "Ergonomic Wireless Mouse", "price": 900, "stock": 5, "category": "wireless mouse"},
    "P003": {"name": "Standard Keyboard", "price": 700, "stock": 8, "category": "keyboard"},
    "P004": {"name": "Mechanical Keyboard", "price": 1800, "stock": 3, "category": "keyboard"},
    "P005": {"name": "USB Webcam", "price": 1200, "stock": 0, "category": "webcam"},
}

ORDER_FILE = Path("/content/mock_orders.json")
print("Catalogue loaded:", len(PRODUCTS), "products")
for pid, p in PRODUCTS.items():
    print(f"{pid} | {p['name']} | ₹{p['price']:,} | stock={p['stock']}")

Catalogue loaded: 5 products
P001 | Basic Wireless Mouse | ₹500 | stock=10
P002 | Ergonomic Wireless Mouse | ₹900 | stock=5
P003 | Standard Keyboard | ₹700 | stock=8
P004 | Mechanical Keyboard | ₹1,800 | stock=3
P005 | USB Webcam | ₹1,200 | stock=0


## 4. Python tools: search, stock, totals, and safe order saving
All financial and stock decisions are enforced in Python rather than trusted to the language model.

In [4]:
def load_orders():
    if not ORDER_FILE.exists():
        return []
    with ORDER_FILE.open("r", encoding="utf-8") as f:
        return json.load(f)


def search_products(query=""):
    query = query.lower().strip()
    query = query.replace("mice", "mouse").replace("keyboards", "keyboard")
    matches = []
    for pid, product in PRODUCTS.items():
        searchable = f"{pid} {product['name']} {product['category']}".lower()
        if not query or query in searchable:
            matches.append({
                "product_id": pid,
                **product,
                "available": product["stock"] > 0,
            })
    return matches


def check_stock(product_id):
    if product_id not in PRODUCTS:
        return {"product_id": product_id, "in_stock": False, "available_stock": 0,
                "error": "Unknown product ID"}
    p = PRODUCTS[product_id]
    return {
        "product_id": product_id,
        "name": p["name"],
        "available_stock": p["stock"],
        "in_stock": p["stock"] > 0,
    }


def calculate_total(items):
    if not isinstance(items, list) or not items:
        raise ValueError("Choose at least one product.")
    total, lines, seen = 0, [], set()
    for item in items:
        if not isinstance(item, dict):
            raise ValueError("Each item must contain product_id and quantity.")
        pid, qty = item.get("product_id"), item.get("quantity")
        if pid not in PRODUCTS:
            raise ValueError(f"Unknown product ID: {pid}")
        if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
            raise ValueError("Quantities must be positive whole numbers.")
        if pid in seen:
            raise ValueError("A product may appear only once; combine its quantity.")
        seen.add(pid)
        p = PRODUCTS[pid]
        if p["stock"] <= 0:
            raise ValueError(f"{p['name']} is out of stock.")
        if qty > p["stock"]:
            raise ValueError(f"Only {p['stock']} unit(s) of {p['name']} are available.")
        subtotal = qty * p["price"]
        total += subtotal
        lines.append({
            "product_id": pid, "product": p["name"], "quantity": qty,
            "unit_price": p["price"], "subtotal": subtotal
        })
    return {"items": lines, "total": total}


def place_order(items, approval, preview_id, budget=None):
    # Approval must be an explicit boolean True, not a truthy string.
    if approval is not True:
        return {"status": "rejected", "message": "No order was saved."}
    if not preview_id:
        raise ValueError("Missing preview ID; prepare a fresh preview first.")
    calculated = calculate_total(items)
    if budget is not None and calculated["total"] > budget:
        raise ValueError("Order exceeds the stated budget.")
    orders = load_orders()
    for order in orders:
        if order.get("preview_id") == preview_id:
            return {"status": "already_saved", "order_id": order["order_id"],
                    "message": "This approved preview was already saved."}
    order = {
        "order_id": "ORD-" + uuid.uuid4().hex[:10].upper(),
        "preview_id": preview_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "items": calculated["items"],
        "total": calculated["total"],
        "budget": budget,
        "status": "confirmed",
    }
    orders.append(order)
    # Write to a temporary file first, then replace, to reduce partial-write risk.
    tmp = ORDER_FILE.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(orders, f, indent=2, ensure_ascii=False)
    tmp.replace(ORDER_FILE)
    return {"status": "success", "order_id": order["order_id"],
            "total": calculated["total"], "message": "Mock order saved."}

print("Python tools ready.")

Python tools ready.


## 5. Gemini + LangGraph workflow
Gemini interprets natural-language requests and maps them to catalogue IDs. Python validates the proposed items. LangGraph pauses at the approval node and resumes only after the customer's response.

In [5]:
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

class ShoppingState(TypedDict, total=False):
    input_text: str
    action: str
    query: str
    items: list
    budget: Optional[int]
    needs_clarification: bool
    assistant_message: str
    preview: dict
    preview_id: str
    approval_decision: str
    order_result: dict


def _response_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        # Gemini integrations may return text blocks rather than a plain string.
        return "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content)


def parse_json_response(content):
    text = _response_text(content).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    # If extra text surrounds JSON, take the outermost JSON object.
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("Gemini did not return a JSON object. Please try again.")
    return json.loads(text[start:end + 1])


def understand_request(state: ShoppingState):
    old_items = state.get("items", [])
    old_budget = state.get("budget")
    user_text = state["input_text"]
    catalogue = [
        {"product_id": pid, "name": p["name"], "price": p["price"],
         "stock": p["stock"], "category": p["category"]}
        for pid, p in PRODUCTS.items()
    ]
    prompt = f'''
You are the intent parser for a fictional shopping assistant.
Return only JSON with keys: action, query, items, budget, needs_clarification, reply.
action must be "browse", "shop", or "clarify".
items is a list of objects with product_id and positive integer quantity.
Only use product IDs from the catalogue. Never invent products.
For browsing requests, set action="browse" and query to the requested category/name.
For purchase requests, map the requested products to IDs.
For "wireless mouse and keyboard within ₹2000", select the cheapest available
mouse and cheapest available keyboard that fit the budget.
If the customer asks for an unavailable product, do not silently substitute it;
explain it is out of stock and offer available alternatives.
For follow-ups such as "use the cheaper mouse instead", preserve unaffected
items and change only the requested item. Keep the prior budget unless changed.
If the customer requests a purchase but gives no budget or preferences, ask a
clarifying question and set needs_clarification=true.
Catalogue: {json.dumps(catalogue, ensure_ascii=False)}
Previous items: {json.dumps(old_items, ensure_ascii=False)}
Previous budget: {json.dumps(old_budget)}
Customer message: {user_text}
'''
    response = llm.invoke(prompt)
    data = parse_json_response(response.content)

    action = data.get("action", "clarify")
    if action not in {"browse", "shop", "clarify"}:
        action = "clarify"

    items = data.get("items")
    if items is None:
        items = old_items
    budget = data.get("budget", old_budget)
    if budget is not None and (
        isinstance(budget, bool) or not isinstance(budget, int) or budget < 0
    ):
        budget = old_budget

    # Normalize product IDs and quantities; invalid output is rejected by Python later.
    if not isinstance(items, list):
        items = []

    return {
        "action": action,
        "query": str(data.get("query") or ""),
        "items": items,
        "budget": budget,
        "needs_clarification": bool(data.get("needs_clarification", False)),
        "assistant_message": str(data.get("reply") or ""),
        # Clear stale approval/preview/order state on every new customer turn.
        "preview": {},
        "preview_id": "",
        "approval_decision": "",
        "order_result": {},
    }


def browse_products(state: ShoppingState):
    matches = search_products(state.get("query", ""))
    if not matches:
        return {"assistant_message": "No matching products found. Try 'mouse', 'keyboard', or 'webcam'."}
    lines = ["### Matching products"]
    for p in matches:
        stock_text = f"In stock: {p['stock']}" if p["stock"] else "OUT OF STOCK"
        lines.append(f"- **{p['name']} ({p['product_id']})** — ₹{p['price']:,} · {stock_text}")
    return {"assistant_message": "\n".join(lines)}


def prepare_order(state: ShoppingState):
    items, budget = state.get("items", []), state.get("budget")
    if state.get("needs_clarification"):
        return {"assistant_message": state.get("assistant_message") or
                "What is your budget, or do you have a product preference?"}
    if not items:
        return {"assistant_message": state.get("assistant_message") or
                "Which product would you like, and how many units?"}
    try:
        calculated = calculate_total(items)
    except (ValueError, TypeError, KeyError) as exc:
        return {"assistant_message": f"I can't prepare that order: {exc}"}

    if budget is not None and calculated["total"] > budget:
        total = calculated["total"]
        return {"assistant_message":
                f"Your selected items cost ₹{total:,}, which is ₹{total-budget:,} over your ₹{budget:,} budget. "
                "Try the Basic Wireless Mouse (₹500) and Standard Keyboard (₹700), or tell me what you'd like to change."}

    lines = ["### Order preview", "", "| Product | Qty | Unit price | Subtotal |",
             "|---|---:|---:|---:|"]
    for line in calculated["items"]:
        lines.append(f"| {line['product']} | {line['quantity']} | ₹{line['unit_price']:,} | ₹{line['subtotal']:,} |")
    total = calculated["total"]
    lines += ["", f"**Total: ₹{total:,}**", "**Delivery: Free**", "**Taxes: Included**"]
    if budget is not None:
        lines += [f"**Budget: ₹{budget:,}**", f"**Remaining budget: ₹{budget-total:,}**"]
    lines += ["", "Reply `approve` to place the mock order, `reject` to cancel, or `edit: ...` to change it."]
    return {"preview": calculated, "preview_id": uuid.uuid4().hex,
            "assistant_message": "\n".join(lines)}


def approval_node(state: ShoppingState):
    decision = interrupt({
        "message": state["assistant_message"],
        "preview": state.get("preview", {}),
        "budget": state.get("budget"),
        "choices": ["approve", "reject", "edit: <your changes>"],
    })
    if not isinstance(decision, str):
        decision = "reject"
    decision = decision.strip()
    low = decision.lower()
    if low in {"approve", "yes", "confirm", "confirmed"}:
        return {"approval_decision": "approve"}
    if low in {"reject", "no", "cancel"}:
        return {"approval_decision": "reject"}
    if low.startswith("edit:") and decision[5:].strip():
        return {"approval_decision": "edit", "input_text": decision[5:].strip()}
    return {"approval_decision": "reject",
            "assistant_message": "Approval was not recognized. No order was placed."}


def save_order_node(state: ShoppingState):
    try:
        result = place_order(
            items=state["items"], approval=True,
            preview_id=state["preview_id"], budget=state.get("budget")
        )
        if result["status"] in {"success", "already_saved"}:
            message = (f"### Order confirmation\n\n**Order ID:** {result['order_id']}\n\n"
                       f"**Total:** ₹{result.get('total', state['preview']['total']):,}\n\n"
                       f"Status: {result['status']}")
        else:
            message = result["message"]
    except (ValueError, TypeError, KeyError) as exc:
        result = {"status": "error", "message": str(exc)}
        message = f"Order was not saved: {exc}"
    return {"order_result": result, "assistant_message": message}


def reject_order_node(state: ShoppingState):
    return {"order_result": {"status": "rejected", "message": "No order was saved."},
            "assistant_message": "Order cancelled. No order was saved. You can choose different products."}


def route_after_understanding(state: ShoppingState):
    if state.get("action") == "browse":
        return "browse"
    if state.get("action") == "clarify":
        return "clarify"
    return "prepare"


def route_after_preview(state: ShoppingState):
    return "approval" if state.get("preview") else "done"


def route_after_approval(state: ShoppingState):
    decision = state.get("approval_decision")
    if decision == "approve":
        return "save"
    if decision == "edit":
        return "understand"
    return "reject"


builder = StateGraph(ShoppingState)
builder.add_node("understand", understand_request)
builder.add_node("browse", browse_products)
builder.add_node("prepare", prepare_order)
builder.add_node("approval", approval_node)
builder.add_node("save", save_order_node)
builder.add_node("reject", reject_order_node)
builder.add_node("clarify", lambda s: {"assistant_message": s.get("assistant_message") or "What budget or product preference should I use?"})
builder.add_edge(START, "understand")
builder.add_conditional_edges("understand", route_after_understanding,
    {"browse": "browse", "clarify": "clarify", "prepare": "prepare"})
builder.add_edge("browse", END)
builder.add_edge("clarify", END)
builder.add_conditional_edges("prepare", route_after_preview, {"approval": "approval", "done": END})
builder.add_conditional_edges("approval", route_after_approval,
    {"save": "save", "reject": "reject", "understand": "understand"})
builder.add_edge("save", END)
builder.add_edge("reject", END)

graph = builder.compile(checkpointer=InMemorySaver())
print(f"LangGraph workflow compiled. Gemini model configured: {GEMINI_MODEL}")

LangGraph workflow compiled. Gemini model configured: gemini-3.8-flash


## 6. Interactive chat
The same thread preserves conversation context during the runtime. Every edited order gets a fresh preview and must be approved again.

In [10]:

import ipywidgets as widgets
from IPython.display import display, Markdown

THREAD_ID = str(uuid.uuid4())
CONFIG = {"configurable": {"thread_id": THREAD_ID}}

output = widgets.Output()
message_box = widgets.Text(
    placeholder="Ask for products or change your order...",
    description="You:",
    layout=widgets.Layout(width="75%")
)
send_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="send"
)

def show_message(message):
    with output:
        display(Markdown(message))

def handle_result(result):
    show_message(result.get("assistant_message", ""))

    if result.get("__interrupt__"):
        approve_btn = widgets.Button(
            description="Approve order",
            button_style="success",
            icon="check"
        )
        reject_btn = widgets.Button(
            description="Reject order",
            button_style="danger",
            icon="times"
        )
        edit_box = widgets.Text(
            placeholder="Example: use the cheaper mouse instead",
            layout=widgets.Layout(width="350px")
        )
        edit_btn = widgets.Button(
            description="Edit order",
            icon="pencil"
        )

        controls = widgets.VBox([
            widgets.HTML("<b>Confirm your order:</b>"),
            widgets.HBox([approve_btn, reject_btn]),
            widgets.HTML("<b>Or edit your order:</b>"),
            widgets.HBox([edit_box, edit_btn])
        ])

        show_message("Choose an option below:")
        with output:
            display(controls)

        def resume_order(decision):
            approve_btn.disabled = True
            reject_btn.disabled = True
            edit_btn.disabled = True

            try:
                next_result = graph.invoke(
                    Command(resume=decision),
                    config=CONFIG
                )
                handle_result(next_result)
            except Exception as exc:
                show_message(f"**Error:** {exc}")

        approve_btn.on_click(
            lambda _: resume_order("approve")
        )
        reject_btn.on_click(
            lambda _: resume_order("reject")
        )
        edit_btn.on_click(
            lambda _: resume_order(
                "edit: " + edit_box.value
            ) if edit_box.value.strip()
            else show_message("Please enter your changes first.")
        )

    elif result.get("order_result"):
        show_message(
            "**Order status:** "
            + result["order_result"].get("status", "unknown")
        )

def send_message(_):
    text = message_box.value.strip()
    if not text:
        return

    message_box.value = ""
    send_button.disabled = True

    try:
        show_message(f"**You:** {text}")
        result = graph.invoke(
            {"input_text": text},
            config=CONFIG
        )
        handle_result(result)
    except Exception as exc:
        show_message(f"**Error:** {type(exc).__name__}: {exc}")
    finally:
        send_button.disabled = False

send_button.on_click(send_message)
message_box.on_submit(send_message)

display(widgets.VBox([
    widgets.HTML("<h3>AI Shopping Assistant</h3>"),
    widgets.HBox([message_box, send_button]),
    output
]))


## 7. Offline validation of the Python tools
These checks do not call Gemini and do not create orders. They verify arithmetic, stock, invalid quantities, and rejection safety.

In [7]:
# These tests are safe to rerun and do not save any order.
assert search_products("wireless mice")
assert check_stock("P005")["in_stock"] is False
assert calculate_total([{"product_id": "P001", "quantity": 1},
                        {"product_id": "P003", "quantity": 1}])["total"] == 1200

try:
    calculate_total([{"product_id": "P005", "quantity": 1}])
    raise AssertionError("Out-of-stock product should have failed.")
except ValueError:
    pass

try:
    calculate_total([{"product_id": "P001", "quantity": 0}])
    raise AssertionError("Zero quantity should have failed.")
except ValueError:
    pass

before = len(load_orders())
rejected = place_order(
    [{"product_id": "P001", "quantity": 1}],
    approval=False, preview_id="test-rejected-preview"
)
assert rejected["status"] == "rejected"
assert len(load_orders()) == before

print("PASS: search, stock, total, quantity validation, and rejection safety.")

PASS: search, stock, total, quantity validation, and rejection safety.


## 8. Manual acceptance tests
Try these in the chat cell:
1. `Show me wireless mice.`
2. `I need one wireless mouse and one keyboard within ₹2000.`
3. At the preview, type `reject` and verify no order was added.
4. Make another request, then type `approve` and verify an order ID is returned.
5. `I want the USB webcam.` — it should explain that it is out of stock.
6. At a preview, type `edit: use the cheaper mouse instead` — review the new preview and approve it separately.
7. Repeat an approval submission for the same preview in a direct test: the `preview_id` guard prevents duplicate JSON orders.

**Persistence note:** `/content/mock_orders.json` remains available only while the Colab runtime storage survives. Download it or mount Google Drive if you need persistence across runtime resets. The in-memory LangGraph checkpoint also resets with the runtime.